# CCP Literacy Estimation — Step-by-Step Demo (Model 1)

This notebook shows CCP estimation, transitions, CCS, a small MD fit, and a light EM loop.

In [ ]:
import numpy as np, pandas as pd, json
from collections import defaultdict
CSV_PATH = '/mnt/data/toy_panel.csv'
BETA=0.96
X_GRID = np.array([0.0,0.25,0.5,0.75,1.0])
J=len(X_GRID)
A_BINS,Y_BINS,AGE_BINS=6,4,4
ALPHA=5.0
H,DRAWS=3,30
K_TYPES=2

In [ ]:
df = pd.read_csv(CSV_PATH)
df.head()

In [ ]:
EULER_GAMMA=0.5772156649015329
import numpy as np

def quantile_bins(series, nbins):
    qs = np.linspace(0,1,nbins+1)
    cuts = series.quantile(qs).values.astype(float)
    for i in range(1,len(cuts)):
        if cuts[i] <= cuts[i-1]:
            cuts[i] = cuts[i-1] + 1e-9
    return cuts

def cut_to_bins(x, cuts):
    return int(np.clip(np.searchsorted(cuts, x, side='right')-1, 0, len(cuts)-2))

def dirichlet_smooth(counts, alpha=5.0):
    counts = np.asarray(counts,float)
    prior = alpha/len(counts)
    return (counts + prior)/(counts.sum()+alpha)

def inclusive_value_from_ccp(P):
    P = np.clip(np.asarray(P,float), 1e-12,1-1e-12)
    return -float(np.log(P[0]))

def softmax(v):
    v = np.asarray(v,float); v = v - np.max(v)
    ex = np.exp(v); return ex/ex.sum()

In [ ]:
a_cuts = quantile_bins(df['a'], A_BINS)
y_cuts = quantile_bins(df['y'], Y_BINS)
age_cuts = quantile_bins(df['age'], AGE_BINS)

dfb = df.copy()
dfb['ia'] = dfb['a'].apply(lambda v: cut_to_bins(v, a_cuts))
dfb['iy'] = dfb['y'].apply(lambda v: cut_to_bins(v, y_cuts))
dfb['iage'] = dfb['age'].apply(lambda v: cut_to_bins(v, age_cuts))

def x_to_j(x): 
    import numpy as np
    return int(np.argmin(np.abs(X_GRID - float(x))))
dfb['j'] = dfb['x'].apply(x_to_j)

dfb = dfb.sort_values(['id','t']).reset_index(drop=True)
for col in ['ia','iy','iage','j']:
    dfb[col+'_next'] = dfb.groupby('id')[col].shift(-1)
dfb = dfb.dropna(subset=['ia_next','iy_next','iage_next']).copy()
for col in ['ia_next','iy_next','iage_next','j_next']:
    dfb[col] = dfb[col].astype(int)

dfb.head()

In [ ]:
from collections import defaultdict
J=len(X_GRID)
counts = defaultdict(lambda: np.zeros(J, float))
for _,row in dfb.iterrows():
    s=(int(row['ia']), int(row['iy']), int(row['iage']))
    j=int(row['j'])
    counts[s][j]+=1.0
ccp={}; state_counts={}
for s,cvec in counts.items():
    state_counts[s]=float(cvec.sum())
    ccp[s]=dirichlet_smooth(cvec, alpha=ALPHA)
len(ccp)

In [ ]:
trans = defaultdict(lambda: defaultdict(float))
for _, row in dfb.iterrows():
    s=(int(row['ia']), int(row['iy']), int(row['iage']))
    j=int(row['j'])
    sp=(int(row['ia_next']), int(row['iy_next']), int(row['iage_next']))
    trans[(s,j)][sp]+=1.0

trans_prob={}
for key,d in trans.items():
    items=list(d.items())
    probs=np.array([v for (_,v) in items], float); probs=probs/probs.sum()
    trans_prob[key]=([sp for (sp,_) in items], probs)

def draw_next_state(s,j):
    key=(s,j)
    if key not in trans_prob:
        cand=[(k,v) for (k,v) in trans_prob.items() if k[0]==s]
        if not cand:
            return s
        sps=[]; ps=[]
        for (_, (sp_list, p_list)) in cand:
            sps+=sp_list; ps+=list(p_list/len(cand))
        ps=np.array(ps,float); ps=ps/ps.sum()
        idx=np.random.choice(len(sps), p=ps)
        return sps[idx]
    sps,ps=trans_prob[key]
    idx=np.random.choice(len(sps), p=ps)
    return sps[idx]

In [ ]:
BETA=0.96; H=3; DRAWS=30
states=list(ccp.keys())

def delta_ev_ccs(s,j):
    def path_val(start_s, initial_j):
        vals=[]
        for _ in range(DRAWS):
            s_curr=draw_next_state(start_s, initial_j)
            acc=(BETA**1)*inclusive_value_from_ccp(ccp.get(s_curr, np.ones(J)/J))
            for h in range(2,H+1):
                P_curr=ccp.get(s_curr, None)
                if P_curr is None:
                    j_draw=0
                else:
                    j_draw=int(np.random.choice(J, p=P_curr))
                s_curr=draw_next_state(s_curr, j_draw)
                acc+=(BETA**h)*inclusive_value_from_ccp(ccp.get(s_curr, np.ones(J)/J))
            vals.append(acc)
        return float(np.mean(vals)) if vals else 0.0
    return path_val(s,j) - path_val(s,0)

delta_cache={}
for s in states:
    for j in range(1,J):
        delta_cache[(s,j)] = delta_ev_ccs(s,j)

len(delta_cache)

In [ ]:
a_mids = np.array([0.5*(a_cuts[b]+a_cuts[b+1]) for b in range(len(a_cuts)-1)], float)

def objective(theta):
    k0, kL_low, kL_high, phi_low, phi_high, pi_low = theta
    pi=np.array([pi_low, max(1e-6, 1.0-pi_low)]); pi=pi/np.sum(pi)
    loss=0.0; wsum=0.0
    for s in states:
        ia,iy,iage = s
        P = ccp[s]
        a_mid = a_mids[ia]
        costs = np.array([k0 + kL_low + phi_low*a_mid, k0 + kL_high + phi_high*a_mid])
        exp_cost = float(np.sum(pi * costs))
        for j in range(1,J):
            logodds = float(np.log(P[j]) - np.log(P[0]))
            lam = -exp_cost + BETA * delta_cache[(s,j)]
            w = max(state_counts.get(s,1.0), 1.0)
            loss += (logodds - lam)**2 * w
            wsum += w
    return loss / max(wsum,1.0)

theta0 = np.array([20.0, 20.0, 5.0, 0.01, 0.005, 0.5], float)
rng = np.random.default_rng(0)
best_theta = theta0.copy(); best_val = objective(best_theta)
for it in range(200):
    cand = best_theta + rng.normal(0, [2,2,2,0.001,0.001,0.05], size=6)
    val = objective(cand)
    if val < best_val:
        best_theta, best_val = cand, val
best_val, best_theta

In [ ]:
def type_ccp(theta):
    k0, kL_low, kL_high, phi_low, phi_high, pi_low = theta
    def ccps_at_state(s, L):
        ia,_,_ = s
        a_mid = a_mids[ia]
        if L==0:
            kL, phi = kL_low, phi_low
        else:
            kL, phi = kL_high, phi_high
        lam = np.zeros(J); lam[0]=0.0
        for j in range(1,J):
            lam[j] = - (k0 + kL + phi*a_mid) + BETA * delta_cache[(s,j)]
        return softmax(lam)
    return ccps_at_state

traj = defaultdict(list)
for _, row in dfb.sort_values(['id','t']).iterrows():
    s_t = (int(row['ia']), int(row['iy']), int(row['iage']))
    j_t = int(row['j'])
    traj[int(row['id'])].append((s_t, j_t))

def em(theta_init, n_iter=2):
    theta = theta_init.copy()
    k0, kL_low, kL_high, phi_low, phi_high, pi_low = theta
    pi = np.array([pi_low, max(1e-6, 1.0-pi_low)]); pi = pi/np.sum(pi)
    for it in range(n_iter):
        ccps = type_ccp(theta)
        omega={}
        for pid, seq in traj.items():
            like=np.zeros(2)
            for L in [0,1]:
                ll=0.0
                for (s_t, j_t) in seq:
                    P = ccps(s_t, L)
                    ll += np.log(max(P[j_t], 1e-12))
                like[L]=np.exp(ll)
            post = pi * like
            s = post.sum()
            omega[pid] = post/s if s>0 else np.array([0.5,0.5])
        # update pi
        mat = np.stack(list(omega.values()))
        pi = mat.mean(axis=0); pi = pi/np.sum(pi)
        # small local search to re-fit costs keeping pi fixed
        def objective_fixed_pi(th):
            th = th.copy()
            th[-1] = pi[0]
            return objective(th)
        best = theta.copy(); best[-1] = pi[0]
        best_val = objective(best)
        for _ in range(100):
            cand = best + np.random.normal(0, [1,1,1,0.0005,0.0005,0.0], size=6)
            val = objective_fixed_pi(cand)
            if val < best_val:
                best, best_val = cand, val
        theta = best; theta[-1] = pi[0]
    return theta, pi, omega

theta_em, pi_em, omega = em(best_theta, n_iter=2)
theta_em, pi_em

In [ ]:
rows=[]
for pid, post in omega.items():
    rows.append({'id':pid, 'p_low': float(post[0]), 'p_high': float(post[1])})
post_df = pd.DataFrame(rows).sort_values('id').reset_index(drop=True)
post_df.head()

In [ ]:
summary = {
    'theta_em': list(map(float, theta_em)),
    'pi_em': list(map(float, pi_em)),
    'objective_value': float(objective(theta_em))
}
print(json.dumps(summary, indent=2))